In [1]:
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

from post_processing.additional.loaders import enrich_df_with_whole_who_tree_classification
from post_processing.loaders import path_with_old_content_root
from post_processing.tumour_mapper import load_tumour_mappings_as_df

Current working directory: C:\Users\Janek\Desktop\Projects\stainology-dashboard\experiments\notebooks


In [ ]:
tumour_mappings = load_tumour_mappings_as_df(path_with_old_content_root("stains_data_cleaner/classification_benchmark_full_gpt_5/"),
                                             model_glob_key="*gpt*")
tumour_mappings.loc[tumour_mappings['matched_tumour'] == 'Artificial Taxon', 'matched_tumour'] = 'Artificial taxon'
tumour_mappings.loc[tumour_mappings['matched_tumour'] == 'Metastasis to ovary', 'matched_tumour'] = 'Metastases to the Ovary'
tumour_mappings = enrich_df_with_whole_who_tree_classification(tumour_mappings)
tumour_mappings.tumour = tumour_mappings.tumour.str.strip().str.lower()
tumour_mappings.PMID = tumour_mappings.PMID.str.strip()
tumour_mappings.group = tumour_mappings.group.str.strip().str.lower()
tumour_mappings.group_cat = tumour_mappings.group_cat.str.strip().str.lower()
tumour_mappings.notes = tumour_mappings.notes.str.strip().str.lower()
tumour_mappings

In [3]:
dictionary = pd.read_csv("***")
dictionary = dictionary[~dictionary.Notatki.astype(str).str.lower().str.contains("błąd")]
dictionary = dictionary.iloc[:, 5:14]
dictionary.PMID = dictionary.PMID.str.split("gov/").str[-1]
dictionary['matched_tumour'] = dictionary.iloc[:, 6:9].bfill(axis=1).iloc[:, 0]
# # dictionary.loc[dictionary['matched_tumour'] == 'Artificial Taxon', 'matched_tumour'] = 'Artificial taxon'
# # dictionary.loc[dictionary['matched_tumour'] == 'Metastasis to ovary', 'matched_tumour'] = 'Metastases to the Ovary'
dictionary = dictionary.drop(columns=['tumour_class_gt', 'category_2_gt', 'category_1_gt'])
dictionary = enrich_df_with_whole_who_tree_classification(dictionary)
dictionary = dictionary.dropna(subset=['matched_tumour'], axis=0)
dictionary = dictionary.drop_duplicates(subset=['PMID', 'tumour', 'group', 'group_cat', 'notes'], keep='last')
dictionary

Initial shape: (564, 7)
Final shape after enrichment: (564, 11)


,tumour,group,group_cat,notes,PMID,table_id,matched_tumour,tumour_class,category_2,category_1,mt_level
0,benign serous cystadenomas,benign tumors,serous,NaN,16445626,16445626_abstract,"Serous cystadenoma, adenofibroma, surface papi...","Serous cystadenoma, adenofibroma, surface papi...",Benign Serous Tumours,Serous Tumours,0.0
1,benign mucinous cystadenomas,benign tumors,mucinous,NaN,16445626,16445626_abstract,Mucinous cystadenoma and adenofibroma,Mucinous cystadenoma and adenofibroma,Benign Mucinous Tumours,Mucinous Tumours,0.0
2,invasive ovarian cancer,malignant epithelial ovarian tumors,invasive,NaN,16445626,16445626_abstract,Artificial taxon,Artificial taxon,NaN,NaN,0.0
3,undifferentiated carcinoma,NaN,NaN,NaN,16445626,16445626_page_2_table_0,Undifferentiated and dedifferentiated carcinom...,Undifferentiated and dedifferentiated carcinom...,NaN,Other Carcinomas,0.0
4,serous borderline tumor,NaN,NaN,NaN,16803476,16803476_page_2_table_1,Serous borderline tumour of the ovary,Serous borderline tumour of the ovary,Borderline Serous Tumours,Serous Tumours,0.0
...,...,...,...,...,...,...,...,...,...,...,...
567,paracancer tissues,diagnostic category,paracancer tissues,NaN,36594088,36594088_page_10_table_1,Artificial taxon,Artificial taxon,NaN,NaN,0.0
568,hgsoc-pds,diagnosis,hgsoc-pds,biopsy and tma,37124505,37124505_page_5_table_0,High-grade serous carcinoma of the ovary,High-grade serous carcinoma of the ovary,Malignant Serous Tumours,Serous Tumours,0.0
569,serous/homologous ocs,NaN,NaN,NaN,39716847,39716847_abstract,Carcinosarcoma of the ovary,Carcinosarcoma of the ovary,NaN,Other Carcinomas,0.0
570,pheochromocytoma,NaN,NaN,NaN,9553804,9553804_page_5_table_0,Not Ovarian,Not Ovarian,NaN,NaN,0.0


In [6]:
res = dictionary.merge(tumour_mappings, on=['PMID', 'table_id', 'tumour', 'group', 'group_cat', 'notes'], how='left',
                              suffixes=('_gt', ''))
res = res.dropna(subset=['matched_tumour'], axis=0)
res.PMID.nunique(), res.shape

(181, (535, 17))

In [8]:
res2 = res.copy()

gts, preds = res2.matched_tumour_gt.tolist(), res2.matched_tumour.tolist()
labels = sorted(list(set(gts)))
labels_to_ignore = ["Artificial taxon", "Not Ovarian"]
labels = [label for label in labels if label not in labels_to_ignore]

print("F1 Micro:", f1_score(gts, preds, average='micro', labels=labels))
print("F1 Macro:", f1_score(gts, preds, average='macro', labels=labels, zero_division=0))
print("Accuracy:", accuracy_score(gts, preds))

print("Precision Micro:", precision_score(gts, preds, labels=labels, average='micro'))
print("Precision Macro:", precision_score(gts, preds, labels=labels, average='macro', zero_division=0))

print("Recall Micro:", recall_score(gts, preds, labels=labels, average='micro'))
print("Recall Macro:", recall_score(gts, preds, labels=labels, average='macro', zero_division=0))

# Precision ignoring "Artificial taxon" and "Not Ovarian"
res2_no_artificial = res2[~res2.matched_tumour.isin(labels_to_ignore)].copy()
res2_no_artificial = res2_no_artificial[~res2_no_artificial.matched_tumour_gt.isin(labels_to_ignore)].copy()
print(res2.shape, res2_no_artificial.shape)
gts, preds = res2_no_artificial.matched_tumour_gt.tolist(), res2_no_artificial.matched_tumour.tolist()
print("Precision Micro:", precision_score(gts, preds, labels=labels, average='micro'))
print("Precision Macro:", precision_score(gts, preds, labels=labels, average='macro', zero_division=0))
print("Accuracy:", accuracy_score(gts, preds))
gts, preds = res2.matched_tumour_gt.tolist(), res2.matched_tumour.tolist()



F1 Micro: 0.9556786703601108
F1 Macro: 0.850277716878349
Accuracy: 0.9345794392523364
Precision Micro: 0.9636871508379888
Precision Macro: 0.8690119775508769
Recall Micro: 0.9478021978021978
Recall Macro: 0.8597670250896057
(535, 17) (353, 17)
Precision Micro: 0.9801136363636364
Precision Macro: 0.8818484383000512
Accuracy: 0.9773371104815864


In [92]:
res2 = res.copy()
res2 = res2[res2.mt_level_gt== 0]
# res2.loc[res2['matched_tumour'].isin(['Artificial taxon', 'Not Ovarian']), 'matched_tumour'] = "not found"
# res2.loc[res2['matched_tumour_gt'].isin(['Artificial taxon', 'Not Ovarian']), 'matched_tumour_gt'] = "not found"

gts, preds = res2.matched_tumour_gt.tolist(), res2.matched_tumour.tolist()
labels = sorted(list(set(gts)))
labels_to_ignore = ["Artificial taxon", "Not Ovarian"]
labels = [label for label in labels if label not in labels_to_ignore]
# print(sorted(labels))`
print("F1 Micro:", f1_score(gts, preds, average='micro', labels=labels))
print("F1 Macro:", f1_score(gts, preds, average='macro', labels=labels, zero_division=0))

print("Precision Micro:", precision_score(gts, preds, labels=labels, average='micro'))
print("Precision Macro:", precision_score(gts, preds, labels=labels, average='macro', zero_division=0))

print("Recall Micro:", recall_score(gts, preds, labels=labels, average='micro'))
print("Recall Macro:", recall_score(gts, preds, labels=labels, average='macro', zero_division=0))

# Precision ignoring "Artificial taxon" and "Not Ovarian"
res2_no_artificial = res2[~res2.matched_tumour.isin(labels_to_ignore)].copy()
res2_no_artificial = res2_no_artificial[~res2_no_artificial.matched_tumour_gt.isin(labels_to_ignore)].copy()
print(res2.shape, res2_no_artificial.shape)
gts, preds = res2_no_artificial.matched_tumour_gt.tolist(), res2_no_artificial.matched_tumour.tolist()
print("Precision Micro:", precision_score(gts, preds, labels=labels, average='micro'))
print("Precision Macro:", precision_score(gts, preds, labels=labels, average='macro', zero_division=0))
gts, preds = res2.matched_tumour_gt.tolist(), res2.matched_tumour.tolist()

# # Recall ignoring "Artificial taxon" and "Not Ovarian"
# res2_no_artificial = res2[~res2.matched_tumour_gt.isin(labels_to_ignore)].copy()
# gts, preds = res2_no_artificial.matched_tumour_gt.tolist(), res2_no_artificial.matched_tumour.tolist()
# print("Recall Micro:", recall_score(gts, preds, labels=labels, average='micro'))
# print("Recall Macro:", recall_score(gts, preds, labels=labels, average='macro', zero_division=0))

F1 Micro: 0.9811912225705329
F1 Macro: 0.9401448752334389
Precision Micro: 0.9811912225705329
Precision Macro: 0.9429416210928816
Recall Micro: 0.9811912225705329
Recall Macro: 0.9433106575963719
(490, 17) (314, 17)
Precision Micro: 1.0
Precision Macro: 0.9591836734693877


In [85]:
print(classification_report(gts, preds, labels=labels, zero_division=0))

                                                                  precision    recall  f1-score   support

                                     Adult granulosa cell tumour       0.89      1.00      0.94        24
                                       Borderline Brenner tumour       1.00      1.00      1.00         1
                                                 Brenner Tumours       0.00      0.00      0.00         2
                                                  Brenner tumour       0.50      1.00      0.67         1
                                     Carcinosarcoma of the ovary       1.00      1.00      1.00         9
                                    Clear cell borderline tumour       1.00      1.00      1.00         1
                               Clear cell carcinoma of the ovary       0.95      1.00      0.98        20
                         Clear cell cystadenoma and adenofibroma       1.00      1.00      1.00         3
                                             